# 13.5 Attribute Lookup and Class Creation

**Prerequisites:** 13.1 Objects and Names, 13.4 Where the Memory Goes, 5.1 Python OOPs, 5.4 Protocols  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What `obj.attr` actually does, step by step
- 🔴 **Methods, `@property` and `__slots__` are all descriptors** — one mechanism, three faces
- Data vs non-data descriptors, and the precedence rule that follows
- Writing a validating descriptor with `__set_name__` — how ORMs and `pydantic` work
- `__getattr__` vs `__getattribute__`, and which one you actually want
- 🔴 **`__init_subclass__`** — the modern answer to most metaclass problems
- Metaclasses, last and briefly, with an honest note on when to use one

---

## One mechanism behind three familiar features

**13.4** ended on a detail worth pulling on: adding `__slots__` puts something on the *class*
that occupies the attribute name. That something is a **descriptor**, and once you see it you
find the same machinery behind methods, `@property`, `classmethod`, `staticmethod` and
`functools.cached_property`.

A descriptor is any object defining `__get__`, `__set__` or `__delete__`. When such an object
is found on a class, Python does not hand it to you — it **calls** it.

That single rule explains why `service.ping` gives you a bound method rather than a plain
function, and why `endpoint.port = 70000` can raise.

In [ ]:
class Service:
    __slots__ = ("name",)

    def ping(self):
        return "pong"


ping_attr = Service.__dict__["ping"]
name_attr = Service.__dict__["name"]

print("what is actually stored on the class:")
print(f"  'ping' -> {type(ping_attr).__name__:18} __get__: {hasattr(ping_attr, '__get__')}"
      f"  __set__: {hasattr(ping_attr, '__set__')}")
print(f"  'name' -> {type(name_attr).__name__:18} __get__: {hasattr(name_attr, '__get__')}"
      f"  __set__: {hasattr(name_attr, '__set__')}")

api = Service()
api.name = "api-gateway"

print("\nlooking them up goes through __get__:")
print("  api.ping        ->", api.ping)
print("  api.ping()      ->", api.ping())
print("  api.name        ->", api.name)

print("\nbinding by hand does exactly what attribute access did:")
print("  Service.__dict__['ping'].__get__(api)() ->", ping_attr.__get__(api)())

`ping` is stored as a plain **function**, and a function defines
`__get__`. Looking it up on an instance calls that, which returns a *bound method* with `self`
already attached. There is no special case for methods anywhere in the language — it is the
descriptor protocol doing its job.

`name`, created by `__slots__`, is a **`member_descriptor`** defining both `__get__` and
`__set__`. It reads and writes a fixed offset in the object rather than a dictionary entry,
which is precisely why slotted instances are smaller (**13.4**).

The last line proves the equivalence: calling `__get__` manually produces the same bound
method that `api.ping` produced.

## The lookup order, and why it is what it is

| Kind | Defines | Wins against the instance dict? |
|---|---|---|
| **Data descriptor** | `__set__` or `__delete__` | 🔴 **yes** |
| **Non-data descriptor** | only `__get__` | no |

Reading `obj.attr` walks this order:

1. **Data descriptor** found on `type(obj)` or its MRO (**5.1**) — call its `__get__`
2. The instance's own `__dict__`
3. **Non-data descriptor** or plain class attribute from the MRO
4. `__getattr__`, if the class defines one
5. `AttributeError`

The rule looks arbitrary until you see what it protects: `@property` must be a data descriptor,
or a stray `obj.__dict__["price"] = ...` would silently bypass your validation. Methods are
non-data descriptors, so an instance *can* shadow one — which is what makes monkey-patching a
single object possible.

In [ ]:
class DataDescriptor:
    """Defines __set__, so it is a DATA descriptor."""

    def __get__(self, obj, owner=None):
        return "-> from the DATA descriptor"

    def __set__(self, obj, value):
        obj.__dict__["_stored"] = value


class NonDataDescriptor:
    """Only __get__, so it is a NON-DATA descriptor."""

    def __get__(self, obj, owner=None):
        return "-> from the NON-DATA descriptor"


class Probe:
    guarded = DataDescriptor()
    shadowable = NonDataDescriptor()


probe = Probe()

# Write straight into the instance dict, bypassing normal assignment.
probe.__dict__["guarded"] = "-> from the instance __dict__"
probe.__dict__["shadowable"] = "-> from the instance __dict__"

print("both names now exist on the class AND in the instance dict:\n")
print("  probe.guarded    ", probe.guarded)
print("  probe.shadowable ", probe.shadowable)
print("\n  The data descriptor won; the non-data one was shadowed.")
print("  That is why @property cannot be bypassed, and why you can")
print("  monkey-patch a method onto one instance without touching the class.")

Exactly as the table predicts. The instance dictionary contains both
names, yet only `shadowable` reads from it.

This is the mechanism behind a bug people hit and cannot explain: assigning to an instance
attribute that happens to share a name with a `@property` appears to do nothing. It did
something — it called the property's `__set__`, which very likely raised or redirected.

## Writing one: a validated config field

`@property` is the right tool for one attribute on one class. A **descriptor** is the right
tool when the same validation applies to many fields or many classes — which is exactly the
situation an ORM or a settings library is in.

`__set_name__` (**3.6+**) is what makes this ergonomic: Python calls it at class-creation time
and tells the descriptor which attribute name it was assigned to.

In [ ]:
class Port:
    """A validated TCP port. Reusable across any config class."""

    def __set_name__(self, owner, name):
        # Called automatically when the class body is executed.
        self.public_name = name
        self.private_name = "_" + name

    def __get__(self, obj, owner=None):
        if obj is None:                      # accessed on the class, not an instance
            return self
        return getattr(obj, self.private_name)

    def __set__(self, obj, value):
        if not isinstance(value, int) or isinstance(value, bool):
            raise TypeError(f"{self.public_name} must be an int, got {type(value).__name__}")
        if not 1 <= value <= 65535:
            raise ValueError(f"{self.public_name} must be 1-65535, got {value}")
        setattr(obj, self.private_name, value)


class DatabaseEndpoint:
    port = Port()
    admin_port = Port()                      # same descriptor class, different name

    def __init__(self, host, port, admin_port):
        self.host = host
        self.port = port                     # goes through Port.__set__
        self.admin_port = admin_port


endpoint = DatabaseEndpoint("db-1", 5432, 5433)
print("  valid config      :", endpoint.host, endpoint.port, endpoint.admin_port)
print("  stored privately  :", endpoint.__dict__)

for bad, label in [(70000, "out of range"), ("5432", "a string from an env var")]:
    try:
        endpoint.port = bad
    except (ValueError, TypeError) as exc:
        print(f"  rejected ({label:24}): {type(exc).__name__}: {exc}")

print("\n  __set_name__ gave each instance the right name with no repetition:")
print("   ", DatabaseEndpoint.__dict__["port"].public_name,
      "and", DatabaseEndpoint.__dict__["admin_port"].public_name)

One descriptor class, two fields, no duplicated validation and no
hand-written property pairs. `__set_name__` supplied the names, so `Port()` is written without
arguments and still produces accurate error messages.

🔴 **This is not an academic exercise — it is how the tools you already use are built.**
`pydantic` fields (**18.4**), SQLAlchemy columns (**10.4**) and Django model fields are all
descriptors doing what `Port` does, with more validation behind them. Recognising the pattern is
usually enough; writing your own is occasionally the right call.

> **Reach for a plain `@property` first.** A descriptor earns its keep at the third repetition,
> not the first.

## `__getattr__` vs `__getattribute__`

Two hooks, one letter apart, wildly different risk.

| Hook | Called | Use |
|---|---|---|
| `__getattr__` | 🔴 **only after normal lookup failed** | fallbacks, lazy loading, proxies |
| `__getattribute__` | on **every** attribute access | ⚠️ almost never — trivially recursive |

`__getattr__` is the safe one because it never runs for attributes that exist.

In [ ]:
class Settings:
    """Config object backed by a plain dict, with attribute access as a convenience."""

    def __init__(self, values):
        self._values = values

    def __getattr__(self, name):
        # Only reached when normal lookup has already failed.
        try:
            return self._values[name]
        except KeyError:
            raise AttributeError(
                f"no setting named {name!r} (have: {', '.join(sorted(self._values))})"
            ) from None


settings = Settings({"region": "eu-west-1", "max_retries": 3})

print("  real attribute  :", settings._values["region"])
print("  via __getattr__ :", settings.region, "/", settings.max_retries)

try:
    settings.timeout
except AttributeError as exc:
    print("  missing         : AttributeError:", exc)

print("\n  __getattr__ was never called for _values - that attribute exists,")
print("  so normal lookup found it and the hook stayed out of the way.")

The hook fires only for names that are genuinely absent, so ordinary
attributes cost nothing extra.

⚠️ **The classic `__getattr__` bug** is referring to an attribute inside the hook that does
not exist yet — during `__init__`, or after unpickling. `self._values` is looked up, fails,
calls `__getattr__` again, and recurses until `RecursionError` (**13.2**). If you write one,
reach for `object.__getattribute__(self, "_values")` or guard the name explicitly.

## Class creation: `__init_subclass__` first

A class body is executed, then a class object is built from the result. Two hooks let you
participate, and **you should reach for the simpler one**.

`__init_subclass__` (**3.6+**) runs on the parent every time a subclass is defined. It handles
the overwhelming majority of what metaclasses were historically used for: registration,
validation, and default configuration.

In [ ]:
class Exporter:
    """Base class for report exporters (the kind 19.2 produces)."""

    registry = {}

    def __init_subclass__(cls, /, fmt=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if fmt is None:
            raise TypeError(f"{cls.__name__} must declare fmt=...")
        if fmt in Exporter.registry:
            raise ValueError(f"format {fmt!r} already registered by "
                             f"{Exporter.registry[fmt].__name__}")
        Exporter.registry[fmt] = cls
        cls.fmt = fmt


class JsonExporter(Exporter, fmt="json"):
    def render(self, rows):
        return f"{len(rows)} rows as JSON"


class CsvExporter(Exporter, fmt="csv"):
    def render(self, rows):
        return f"{len(rows)} rows as CSV"


print("  registered automatically:", {f: c.__name__ for f, c in Exporter.registry.items()})
print("  dispatch by format      :", Exporter.registry["csv"]().render([1, 2, 3]))

print("\n  and the validation runs at class-definition time, not at first use:")
try:
    class BrokenExporter(Exporter):          # no fmt=
        pass
except TypeError as exc:
    print("    TypeError:", exc)

try:
    class DuplicateExporter(Exporter, fmt="json"):
        pass
except ValueError as exc:
    print("    ValueError:", exc)

Every subclass registered itself with no decorator, no manual
bookkeeping and no metaclass. Better still, both failures were caught **when the class was
defined** — at import time — rather than at the point some request tried to use it.

🔴 That timing is the real prize. A plugin that forgets its format identifier now breaks the
build, not production. This pattern pairs naturally with the plugin registries in **7.2** and
the exporters in **19.2**.

## Metaclasses, last

A class is an object. The thing that made it is its **metaclass**, and by default that is
`type`. `type` with three arguments builds a class at runtime, which is exactly what the `class`
statement does under the hood.

In [ ]:
class Endpoint:
    pass


print("  type(Endpoint)        :", type(Endpoint).__name__)
print("  type(type(Endpoint))  :", type(type(Endpoint)).__name__)

# The class statement, done by hand.
Runtime = type("Runtime", (), {"origin": "built by type() at runtime",
                               "describe": lambda self: "I am a normal class"})
print("\n  built with type(name, bases, namespace):")
print("   ", Runtime.origin)
print("   ", Runtime().describe())
print("   ", "isinstance of the same machinery:", isinstance(Runtime, type))


class Sealed(type):
    """A metaclass that forbids subclassing anything marked final."""

    def __new__(mcls, name, bases, namespace, **kwargs):
        for base in bases:
            if getattr(base, "_final", False):
                raise TypeError(f"{base.__name__} is final and cannot be subclassed")
        return super().__new__(mcls, name, bases, namespace, **kwargs)


class Ledger(metaclass=Sealed):
    _final = True


print("\n  a metaclass enforcing a rule the class statement cannot:")
try:
    class ForkedLedger(Ledger):
        pass
except TypeError as exc:
    print("    TypeError:", exc)

`Endpoint` is an instance of `type`, and `type` is an instance of
itself — which is where the tower stops.

The `Sealed` metaclass does something `__init_subclass__` genuinely cannot: it rejects the
subclass *before* the class object exists. That is the narrow band where metaclasses still earn
their place.

🔴 **You almost certainly do not need one.** The honest checklist:

| Want to… | Use |
|---|---|
| register, validate or configure subclasses | `__init_subclass__` |
| learn the attribute name a descriptor was given | `__set_name__` |
| add behaviour to one class | a class decorator |
| define an interface | `Protocol` or `ABC` (**5.4**) |
| control class creation itself, or the namespace it is built from | ⚠️ a metaclass |

> **Tim Peters' rule of thumb still holds:** if you are wondering whether you need metaclasses,
> you do not. The people who need them know why. `ABCMeta` and `EnumMeta` (**5.3**) are the
> canonical examples — both in the standard library, both solving problems the simpler hooks
> cannot.

---

## Common Mistakes & Pitfalls

1. 🔴 **Assigning to a name shadowed by a `@property` and expecting it to stick.** A data descriptor always wins over the instance dict.
2. **Writing `__getattribute__` when you meant `__getattr__`.** The former runs on every access and recurses if you touch `self.anything`.
3. 🔴 **Referring to a real attribute inside `__getattr__`.** If it does not exist yet you get infinite recursion, classically during `__init__` or after unpickling.
4. **Forgetting `if obj is None: return self` in `__get__`.** Accessing the descriptor on the *class* then explodes, and `help()` and `inspect` do exactly that.
5. **Storing descriptor state on the descriptor itself.** It is shared by every instance of the owning class — keep per-instance state on the instance.
6. **Omitting `super().__init_subclass__(**kwargs)`.** Cooperative multiple inheritance breaks silently (**5.1**).
7. **Reaching for a metaclass** where `__init_subclass__`, `__set_name__` or a class decorator would do.
8. **Combining metaclasses carelessly.** Two bases with different metaclasses is a `TypeError`, and it is not always obvious which they are.
9. **Assuming `__slots__` names behave like ordinary class attributes.** They are descriptors occupying that name (**13.4**).

## Best Practices

- Use `@property` for one computed or validated attribute; promote to a descriptor at the third repetition.
- Always implement `__set_name__` rather than passing the attribute name in by hand.
- Handle `obj is None` in `__get__` so class-level access and `inspect` keep working.
- Prefer `__getattr__` over `__getattribute__`, and raise `AttributeError` (not `KeyError`) from it.
- Use `__init_subclass__` for registries and validation — it fails at import time, which is where you want it.
- Reach for a class decorator before a metaclass; it composes and a metaclass does not.
- Remember `Protocol` (**5.4**) exists — most "enforce an interface" problems are typing problems (**16.4**).
- Read `type(obj).__mro__` when attribute lookup surprises you; the answer is always in there.

## Practice Exercises

Try these before moving on.

1. Add a `Hostname` descriptor alongside `Port` that rejects empty strings and anything with a scheme. Reuse it in two config classes.
2. 🔴 Write a `__getattr__` that recurses infinitely, watch the `RecursionError` (**13.2**), then fix it with `object.__getattribute__`.
3. Turn `Port` into a non-data descriptor by deleting `__set__`. What breaks, and why does the precedence table predict it?
4. Implement `cached_property` yourself as a non-data descriptor that writes into the instance dict. Explain why it *must* be non-data to work.
5. Extend the `Exporter` registry so a subclass may also declare a file extension, and reject duplicates of that too.
6. Rewrite the `Sealed` metaclass as `__init_subclass__` on `Ledger`. What is different about *when* the error is raised?
7. 🔴 Combine this with **13.4**: give a descriptor-using class `__slots__`, and explain the collision you get between the slot descriptor and your own.
8. Look at `functools.cached_property` and `staticmethod` in the standard library. Which are data descriptors, and does that match their behaviour?
9. **Interview question:** *“Why does `obj.method` return something different from `Class.__dict__['method']`?”* Answer with the descriptor protocol.

---

## Version notes

| Version | Change |
|---|---|
| **3.6** | 🔴 `__set_name__` and `__init_subclass__` — between them these removed most reasons to write a metaclass |
| **3.6** | `__init_subclass__` is implicitly a `classmethod`; you do not write the decorator |
| **3.8** | `functools.cached_property` — a non-data descriptor, which is why it can cache into the instance dict |
| **3.10** | `dataclass(slots=True)` (**13.4**) rebuilds the class, so descriptors defined in the body are re-created |
| **3.12** | PEP 695 type parameters (**16.3**) are handled at class creation without a metaclass |
| **3.12** | `@override` (PEP 698) for checked overriding — a typing answer to an old metaclass use case (**16.4**) |

> ⚠️ **Portability.** The descriptor protocol *is* part of the language and works everywhere.
> The implementation details around it — slot layout, `__dict__` key sharing — are CPython's.

## 13 How Python Works Under the Hood — the folder

| Notebook | Covers |
|---|---|
| **13.1** | objects on the heap, names, identity, interning, immortality |
| **13.2** | the call stack, frames, recursion limits, tracebacks |
| **13.3** | reference counting, cycles, the collector, `weakref` |
| **13.4** | object sizes, container overhead, `__slots__` measured |
| **13.5** | this notebook — attribute lookup, descriptors, class creation |

**The one-sentence version:** *attribute access is a search with one special rule — descriptors
get called instead of returned — and that rule is the whole of methods, properties, slots and
most of what people reach for metaclasses to do.*

## Related

- **5.1** Python OOPs — the MRO this lookup walks
- **5.3** Dataclasses and Enums — `EnumMeta` is a real metaclass doing real work
- **5.4** Duck Typing, Protocols and Composition — the interface answer you usually want
- **10.4 / 18.4** SQLAlchemy and pydantic — descriptors in production
- **13.4** — where the slot descriptors came from
- **16.4** Protocols and Structural Typing — the static-typing view of the same problem